# FrugalProver — H2: does the oracle's budget beat a uniform one?

A **2-hour, 2×RTX-4090** run of the allocation experiment
(`docs/RESEARCH_PLAN.md`, Phase 2). Every arm solves the same problems with the
same agent under the same total token budget; only the split differs.

    oracle -> p̂ᵢ(B) -> [ policy ] -> per-problem caps -> agent

**The clock is the design constraint.** The timings below are budgeted against a
2-hour window, and every step that can fail is checked before the step that
costs an hour. In order:

| min | step |
|---|---|
| 0–15 | install; start the 15GB weight download in the background |
| 15–22 | **§2 extract hidden states — before vLLM takes the cards** |
| 22–27 | §3 refit the oracle with activations, on CPU (E0b) |
| 27–34 | §4 start vLLM, sanity-check token accounting and seeding |
| 34–42 | §5 pilot: acceptance rate, throughput, and size the run |
| 42–105 | §6 the experiment |
| 105–115 | §7 read the report |
| 115–120 | **§8 push the artifacts** |

§8 is not optional. The `label5h` run's `hidden_states.parquet` was lost when
its pod was released, which is why §2 exists at all.

## 1. Install

`.[openai]` is what the driver needs — it talks HTTP and imports no torch.
`.[gpu]` is needed here too, because §2 runs a real forward pass locally.
`vllm` is the server; it shares this runtime's GPUs.

In [ ]:
!git clone https://github.com/newpotatato/smiles-frugalprover.git
%cd smiles-frugalprover
!git checkout agent/oracle

!pip install -q -e ".[gpu,openai]"
!pip install -q vllm

In [ ]:
# Pull the 15GB of weights NOW, in the background, so the download overlaps
# sections 2 and 3 instead of stalling section 4. vLLM will find them cached.
import subprocess

subprocess.Popen(
    ["python", "-c",
     "from huggingface_hub import snapshot_download; "
     "snapshot_download('deepseek-ai/DeepSeek-R1-Distill-Qwen-7B')"],
    stdout=open("/tmp/dl.log", "w"), stderr=subprocess.STDOUT,
)
!frugalprover info

## 2. Hidden states — run this BEFORE the server

Two reasons this is first, not later:

1. **Memory.** vLLM takes 88% of each card and does not give it back. A
   `transformers` extractor started afterwards has nowhere to load.
2. **It is the missing artifact.** `data/label5h/` in the repo has
   `problems.jsonl` and `budgets.jsonl` but no `hidden_states.parquet` — it was
   lost with the pod that produced it (`.gitignore` excludes `data/*/`). Without
   it the oracle cannot be refit with activations at all.

The settings below are copied from `results/deepseek_r1_qwen7b_n3/config.yaml`
and must match it exactly: a fitted oracle carries the scaler and PCA rotation
it was trained with, and those only apply to vectors from the same extractor,
same pooling, same truncation.

~1 minute for the 276 labeled problems; a few more for the eval pool.

In [ ]:
RUN = "label5h"          # the labeled set: problems + budgets are in the repo
EVAL = "h2"              # this experiment's run dir

# Both the labeled problems (to refit the oracle) and the eval pool (to predict
# on) need states. One extraction over the whole 4596-problem file covers both,
# and at 512 input tokens on a 1.5B model it is minutes, not hours.
!frugalprover extract -c configs/base.yaml --run-name {RUN} \
    --set extract.model_name=Qwen/Qwen2.5-Math-1.5B \
    --set extract.all_layers=true --set extract.all_layers_pooling=mean \
    --set extract.max_input_tokens=512 --set extract.dtype=bfloat16 \
    --set extract.batch_size=32 \
    --set extract.problems=label5h/problems.jsonl

## 3. Refit the oracle with activations (E0b) — CPU, no server

This is the half of the offline study that could not be run locally, because it
needs the states from §2. Two questions, both answered in about a minute and
both worth having on disk *before* the GPU-hour is spent:

- Does the activation oracle beat the surface+subject baseline? The committed
  run says no (0.7996 vs 0.8043 in
  `results/deepseek_r1_qwen7b_n3/baselines.json`). Confirming that on a fresh
  extraction is also the check that §2 reproduced the original vectors.
- Does allocation with the activation oracle beat uniform, in simulation?
  The surface-only answer is yes, by ~10 problems out of 276 at `b_bar=2896`.

In [ ]:
!frugalprover train -c configs/base.yaml --run-name {RUN} \
    --set train.problems=label5h/problems.jsonl \
    --set train.budgets=label5h/budgets.jsonl \
    --set train.hidden_states=label5h/hidden_states.parquet

import json
m = json.load(open(f"results/{RUN}/metrics.json"))
print(f"layer {m['layer']}  CV AUC {m['cv_score']}  (committed run: L16_mean, 0.7996)")

In [ ]:
# The allocation simulation, now WITH activations. Compare its verdict against
# the surface-only one already in the repo -- if activations do not move the
# allocation either, that is the same honest negative in a second form.
!python -m frugalprover.analysis.allocation_sim \
    --problems data/{RUN}/problems.jsonl \
    --budgets  data/{RUN}/budgets.jsonl \
    --hidden-states data/{RUN}/hidden_states.parquet \
    --b-bar 2896 --sweep-triage \
    --out-dir results/{RUN}/analysis_activations

## 4. Start vLLM on both cards

`--data-parallel-size 2` runs two independent replicas, one per card. Not
`--tensor-parallel-size`: a 7B fits one 4090 with room for KV cache, so TP would
only add an all-reduce per layer to buy single-request latency, which a batch
job does not care about.

Three flags below are not tuning — the full reasoning is in the header of
`configs/agent/DeepSeek_R1_Distill_Qwen_7B_vllm.yaml`:

- `--served-model-name` must equal `model:` in the config, or every request 404s.
- `--max-model-len 24576`; 20480 is measured-not-estimated too small, and fails
  only once the first repair round is reached — well into the run.
- **never** `--kv-cache-dtype fp8`: measured 7/8 completions collapse into
  repetition loops that run to the full cap.

In [ ]:
import os
import subprocess
import time

import requests

MODEL, SERVED, PORT = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-r1-7b", 8000
BASE, LOG = f"http://127.0.0.1:{PORT}", "/tmp/vllm.log"
os.environ["VLLM_API_KEY"] = "EMPTY"
HEADERS = {"Authorization": f"Bearer {os.environ['VLLM_API_KEY']}"}

N_GPUS = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
             .stdout.strip().count("GPU "))
VRAM_GB = int(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split("\n")[0].strip()) // 1024
GMU, MAX_NUM_SEQS = ("0.88", 24) if VRAM_GB < 40 else ("0.92", 128)
print(f"{N_GPUS} x {VRAM_GB}GB -> --data-parallel-size {N_GPUS}, "
      f"--gpu-memory-utilization {GMU}, --max-num-seqs {MAX_NUM_SEQS} (per replica)")

cmd = ["vllm", "serve", MODEL, "--served-model-name", SERVED,
       "--data-parallel-size", str(N_GPUS), "--dtype", "bfloat16",
       "--gpu-memory-utilization", GMU, "--max-model-len", "24576",
       "--max-num-seqs", str(MAX_NUM_SEQS), "--enable-prefix-caching",
       "--port", str(PORT)]
print(" ".join(cmd))
subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for _ in range(180):                      # patient: the weights may still be landing
    try:
        if requests.get(f"{BASE}/health", timeout=5).status_code == 200:
            print("server up"); break
    except requests.RequestException:
        pass
    time.sleep(10)
else:
    !tail -40 {LOG}
    raise SystemExit("server did not come up -- see the log tail above")

In [ ]:
# Two checks the whole experiment rests on, both cheap.
#
# 1. usage.completion_tokens must be present. It is the ONLY input to the loop's
#    budget cap; without it the client falls back to a character heuristic, the
#    caps fire in the wrong place, and nothing in the output looks wrong.
# 2. The same seed must return the same completion. That is what makes the arms
#    a paired comparison rather than two independent draws -- if vLLM ignores
#    `seed`, agent.seed_mode buys nothing and the run needs more problems to
#    resolve the same effect.
def ask(seed=None, max_tokens=64):
    body = {"model": SERVED, "temperature": 0.6, "top_p": 0.95, "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": "What is 17*23? Put the answer in \\boxed{}."}]}
    if seed is not None:
        body["seed"] = seed
    r = requests.post(f"{BASE}/v1/chat/completions", headers=HEADERS, json=body, timeout=300)
    r.raise_for_status()
    return r.json()

r = ask()
assert (r.get("usage") or {}).get("completion_tokens"), \
    "no usage.completion_tokens -- token accounting would be approximate"
assert not (r["choices"][0]["message"].get("reasoning_content") or ""), \
    "server is splitting out reasoning_content -- restart without --reasoning-parser"

a, b, c = ask(seed=7), ask(seed=7), ask(seed=8)
same = a["choices"][0]["message"]["content"] == b["choices"][0]["message"]["content"]
diff = a["choices"][0]["message"]["content"] != c["choices"][0]["message"]["content"]
print(f"exact token counts : yes ({r['usage']['completion_tokens']} tokens)")
print(f"seed reproducible  : {same}")
print(f"different seed differs: {diff}")
if not same:
    print("\n!! vLLM is not honouring `seed`. The arms will still be valid, just "
          "noisier -- consider --set allocate.max_problems higher, or accept wider "
          "error bars on the paired test.")

## 5. Pilot — is the loop healthy, and how big can the run be?

Eight minutes here protects the hour that follows. Three things it settles:

- **Acceptance rate.** 20–50% is healthy. Near zero means every attempt burns
  its full round budget and the run costs roughly double for nothing —
  `docs/hf_agent.ipynb` §4a has the diagnosis tree for that failure.
- **Throughput.** The `label5h` run measured 512 generated tok/s on one card.
  Two should give ~1000. The measured number sets `max_problems`.
- **The caps actually bind.** Different arms must produce different per-problem
  token totals; if they don't, the budget axis is not doing anything.

In [ ]:
CFG = ("-c configs/base.yaml "
       "-c configs/agent/DeepSeek_R1_Distill_Qwen_7B_vllm.yaml "
       "-c configs/agent/DeepSeek_R1_Distill_Qwen_7B_vllm_2gpu.yaml "
       "-c configs/allocate/h2_matched_compute.yaml")

import time
t0 = time.time()
!frugalprover allocate {CFG} --run-name pilot \
    --set allocate.problems=label5h/problems.jsonl \
    --set allocate.hidden_states=label5h/hidden_states.parquet \
    --set allocate.oracle=label5h/oracle.joblib \
    --set allocate.exclude_labeled=label5h/budgets.jsonl \
    --set allocate.chunk_size=12 --set allocate.time_budget_s=null \
    --max-problems 12
PILOT_S = time.time() - t0
print(f"\npilot wall time: {PILOT_S:.0f}s")

In [ ]:
import collections
import json

from frugalprover.common.io import read_jsonl

rows = read_jsonl("data/pilot/alloc.jsonl")
arms = sorted({r["arm"] for r in rows})
tokens = sum(r["tokens"] for r in rows)

print(f"{'arm':<16}{'caps':>28}{'tokens':>9}{'solved':>8}{'accept':>8}")
for a in arms:
    mine = [r for r in rows if r["arm"] == a]
    caps = dict(sorted(collections.Counter(r["cap"] for r in mine).items()))
    audited = [r for r in mine if r["accepted"] is not None]
    acc = sum(bool(r["accepted"]) for r in audited) / len(audited) if audited else float("nan")
    print(f"{a:<16}{str(caps):>28}{sum(r['tokens'] for r in mine):>9}"
          f"{sum(r['solved'] for r in mine):>8}{acc:>8.0%}")

TOK_PER_S = tokens / PILOT_S
TOK_PER_PROBLEM = tokens / 12            # summed across every arm
print(f"\nthroughput      : {TOK_PER_S:,.0f} generated tok/s")
print(f"cost per problem: {TOK_PER_PROBLEM:,.0f} tokens across {len(arms)} arms")

# Size the run from what was just measured, not from the plan.
SOLVE_MINUTES = 60
MAX_PROBLEMS = int(SOLVE_MINUTES * 60 * TOK_PER_S / TOK_PER_PROBLEM)
TIME_BUDGET_S = SOLVE_MINUTES * 60
print(f"\n-> {SOLVE_MINUTES} solving minutes affords ~{MAX_PROBLEMS} problems")

per_arm_tokens = {a: sum(r["tokens"] for r in rows if r["arm"] == a) for a in arms}
if len(set(per_arm_tokens.values())) == 1:
    print("\n!! every arm spent identically -- the caps are not binding. Check that "
          "the agent client is `openai` and not `mock`.")

## 6. The experiment

Every arm gets `B_tot = b_bar × n_problems` tokens; only the split differs.
`b_bar = 2896` is chosen **in advance** from the offline simulation, not after
seeing these results — at an on-grid `b_bar` (2048, 4096) every arm can afford
the same cap for everyone and the arms come out within noise of each other.
`configs/allocate/h2_matched_compute.yaml` carries the table.

The run is chunk-interleaved: every arm finishes a chunk before the next chunk
starts, so if `time_budget_s` truncates it, all arms hold the same prefix of
problems and the paired comparison survives. It also checkpoints per chunk and
resumes, so rerunning the same cell after a hiccup continues rather than
restarts.

In [ ]:
!frugalprover allocate {CFG} --run-name {EVAL} \
    --set allocate.problems=label5h/problems.jsonl \
    --set allocate.hidden_states=label5h/hidden_states.parquet \
    --set allocate.oracle=label5h/oracle.joblib \
    --set allocate.exclude_labeled=label5h/budgets.jsonl \
    --set allocate.max_problems={MAX_PROBLEMS} \
    --set allocate.time_budget_s={TIME_BUDGET_S}

In [ ]:
# Watch from a second cell while the run proceeds (interrupt when done).
# num_requests_running should sit near --max-num-seqs x replicas. If it is far
# below, the client is not keeping the GPUs fed -- raise allocate.chunk_size.
import re

import requests

m = requests.get(f"{BASE}/metrics", timeout=10).text
for k in ("num_requests_running", "num_requests_waiting", "gpu_cache_usage_perc"):
    hit = re.search(rf"^vllm:{k}\S* ([0-9.eE+-]+)$", m, re.M)
    print(f"{k:<26}{hit.group(1) if hit else '?'}")

## 7. Read the result

The honest reading, in order:

1. **Did an oracle arm beat `uniform`?** That is H2.
2. **Did it beat `length`?** If not, there is no result — longer problems
   plausibly need more tokens for reasons that have nothing to do with reasoning
   difficulty, and that is the bar the README sets.
3. **What did abstention cost?** `oracle_triage` skipping problems the `uniform`
   arm went on to solve is the price of the tokens it freed.
4. **Is the difference bigger than the noise?** The paired McNemar counts are
   the discordant pairs; at ~200 problems a handful of them is not a finding.

In [ ]:
import json

report = json.load(open(f"results/{EVAL}/allocation.json"))
from frugalprover.allocate import metrics

print(metrics.format_table(report))
print(f"\nscored {report['n_scored']} problems in every arm "
      f"({report['n_dropped_partial']} dropped as partial)")
for a in report.get("abstention", []):
    print(f"{a['arm']}: skipped {a['n_skipped']}, of which uniform solved "
          f"{a['skipped_but_baseline_solved']}")

In [ ]:
# Tokens-vs-solved, the success-vs-compute view of the same rows.
import matplotlib.pyplot as plt

arms = report["arms"]
fig, ax = plt.subplots(figsize=(6, 4.5))
for a in arms:
    ax.scatter(a["tokens"] / 1e6, a["n_solved"], s=70)
    ax.annotate(a["arm"], (a["tokens"] / 1e6, a["n_solved"]),
                textcoords="offset points", xytext=(6, 3), fontsize=9)
ax.set_xlabel("generated tokens actually spent (millions)")
ax.set_ylabel(f"problems solved (of {report['n_scored']})")
ax.set_title(f"Matched compute: B_tot = {report['b_bar']} x {report['n_scored']}")
fig.tight_layout()
fig.savefig(f"results/{EVAL}/matched_compute.png", dpi=150)
plt.show()

## 8. Push the artifacts — before the pod is released

`data/*/` and `results/*/` are gitignored, so everything produced here has to be
force-added. The `label5h` hidden states were lost exactly this way.

The full parquet is ~50MB of float32 across 29 layers. Only the chosen layer is
needed to reuse the oracle, and float16 is plenty for a PCA projection, so the
slice below is a couple of MB — small enough to commit, which is the difference
between reproducible next month and re-rentable next month.

In [ ]:
import pandas as pd

LAYER = json.load(open(f"results/{RUN}/metrics.json"))["layer"]
df = pd.read_parquet(f"data/{RUN}/hidden_states.parquet")
slim = df[["id", LAYER]].copy()
slim[LAYER] = slim[LAYER].map(lambda v: pd.array(v, dtype="float16").tolist())
slim.to_parquet(f"data/{RUN}/hidden_states_{LAYER}.parquet", index=False)
print(f"{LAYER} slice: {len(slim)} rows, "
      f"{__import__('os').path.getsize(f'data/{RUN}/hidden_states_{LAYER}.parquet')/1e6:.1f} MB")

In [ ]:
!git config user.email "you@example.com" && git config user.name "H2 run"
!git checkout -b results/h2_allocation

# Artifacts, not raw generations: alloc.jsonl carries every candidate solution
# and runs to tens of MB. The report, the log and the label-side inputs are what
# make the run readable later.
!git add -f results/{EVAL}/allocation.json results/{EVAL}/run.log \
           results/{EVAL}/matched_compute.png \
           results/{RUN}/metrics.json results/{RUN}/oracle.joblib \
           results/{RUN}/analysis_activations/ \
           data/{RUN}/hidden_states_*.parquet data/{RUN}/hidden_states.parquet.meta.json \
           data/{EVAL}/alloc.jsonl.meta.json
!git commit -q -m "results: H2 matched-compute allocation run on 2x4090"
!git push -u origin results/h2_allocation
!git log --oneline -1

### If there is time left

In rough order of value:

1. **Raise `max_problems` and rerun.** Same command — it resumes and extends,
   and more problems is the only thing that tightens the paired test.
2. **A second `b_bar`.** One more operating point turns a point estimate into
   the beginnings of the success-vs-compute curve H2 is actually about
   (`--run-name h2_b4096 --set allocate.b_bar=4096`).
3. **More Stage 2 labels.** 4320 of the 4596 sampled problems are still
   unlabelled, and every one of them makes the next oracle better:
   `frugalprover budget -c configs/base.yaml -c <vllm cfg> --run-name label5h`
   resumes the original sweep where it stopped.